# Zero-LLM T0 Envelope Gate + QCN Path-Consistency Engine

**Artifact:** a reusable, unit-validated **Qualitative Constraint Network (QCN) path-consistency
engine** plus a **zero-LLM-spend "T0 envelope gate"** that decides — *from gold temporal graphs
alone, before any LLM budget is spent* — whether an iterated path-consistency *closure certificate*
is viable on real text, and on which corpus.

### The method vs. baselines (one pipeline)
For every held-out gold edge `(u,v)` we compare three closure variants:

- `predict_our_method_full_pc` — **OUR METHOD**: FULL iterated Mackworth **PC-2** closure to fixpoint.
- `predict_baseline_naive_singlepass` — **BASELINE**: a single pass of length-2 intersections at the
  query edge (Path-of-Thoughts-style).
- `predict_baseline_closure_off` — **lower baseline**: no propagation (returns the universe).

### The T0 funnel (per held-out gold edge)
`evaluable` → **(i)** deduction-required → **(ii)** multi-path → **(iii)** bite-after-widening →
**(iv) N\*** (full PC recovers the exact gold relation).

### This demo
The full artifact parses three real corpora (MATRES, TDDMan, NarrativeTime). To keep the demo fast we
load **pre-parsed gold graphs from one corpus — NarrativeTime** (its EXACT convex-**point** arm and
its sound-lower-bound **Allen** interval arm) — bundled with the authoritative qualreas algebra
tables in `mini_demo_data.json`. The engine unit-test battery (which gates everything and includes
the FULL-vs-NAIVE iteration-isolation proof) runs in full. Config values start at the smallest values
that produce output and can be scaled back up to the original full-run settings (see the config cell).

> The code below is the original `engine.py` / `tests.py` / `method.py`, split into cells with
> explanations. Only the raw-corpus *parsing* stage is replaced by loading pre-parsed gold graphs.


## 1. Install dependencies

Installs `loguru` everywhere; `numpy`/`scipy`/`matplotlib` are pinned to Colab's versions when running locally (skipped on Colab to avoid C-extension ABI breakage).

In [ ]:
# --- Dependencies (works on both Colab and a clean local Jupyter) ---
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru -- NOT pre-installed on Colab, install everywhere
_pip('loguru==0.7.3')

# numpy / scipy / matplotlib -- pre-installed on Colab; install locally to match Colab's versions
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')


## 2. Imports

The original import block (stdlib + numpy/scipy/loguru) plus matplotlib for the final visualization, and a loguru stdout sink.

In [ ]:
# --- Imports (original import block from engine.py / tests.py / method.py + matplotlib for the demo) ---
from __future__ import annotations

import gc
import json
import re
import sys
import time
import random
import itertools
from collections import Counter, defaultdict, deque
from pathlib import Path
from typing import Iterable

import numpy as np
from scipy.stats import norm
from loguru import logger
import matplotlib.pyplot as plt

# loguru -> stdout (the original also adds a rotating file sink; not needed for the demo)
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")


## 3. Load the demo data

Loads `mini_demo_data.json` from GitHub (with a local fallback). It bundles the authoritative **qualreas algebra tables** and the **NarrativeTime gold graphs** (point + Allen views).

In [ ]:
# --- Data loading: GitHub raw URL with a local fallback (Colab-compatible) ---
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-40a89b-no-derivation-no-relation-a-closure-cert/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

import os


In [ ]:
data = load_data()

# The engine's qualreas loader reads algebra tables from files, so materialise the bundled tables
# into ./algebras/ -- this lets the original engine/tests code run completely unchanged.
ALG_DIR = Path("algebras"); ALG_DIR.mkdir(exist_ok=True)
for _name, _tbl in data["algebras"].items():
    (ALG_DIR / f"{_name}.json").write_text(json.dumps(_tbl))

def load_arm(name):
    """Return a pre-parsed gold-graph arm (replaces parsers.py raw-corpus parsing for the demo)."""
    return data["arms"][name]

print("algebra tables:", list(data["algebras"]))
print("gold-graph arms:", {k: len(v["docs"]) for k, v in data["arms"].items()},
      "(corpus:", data.get("source_corpus"), ")")


## 4. Config

Every tunable parameter lives here. Values start small for a fast demo; the trailing comments give the original full-run settings, and `PRE_REG` holds the artifact's pre-registered thresholds verbatim.

In [ ]:
# =====================================================================================
# CONFIG -- all tunable parameters. Values start at the SMALLEST that produce output and
# scale up toward the original full-run settings (shown in the trailing comments).
# =====================================================================================

# --- demo scale (the only knob scaled down for the demo is the corpus subset) ---
# Start tiny to smoke-test, then scale up. The whole pipeline runs in ~10s at the full 25-doc
# subset, so we keep the original full-run settings for everything except the corpus size, which
# is inherently the demo's curated subset (the full artifact runs over the entire NarrativeTime
# corpus + MATRES + TDDMan).
MAX_DOCS_PER_ARM     = 25     # docs processed per arm   (demo subset = 25; original: ALL docs)
# MAX_DOCS_PER_ARM   = 2      # <- absolute-minimum smoke-test value
NODE_CAP             = 200    # skip gold graphs with > this many nodes              (original: 200)
VERIFY_CAP_PER_DOC   = 25     # genuine-PC cross-checks of the complete-graph fast path (original: 25)

# --- engine test battery ---
N_NETS_COMPLETENESS  = 200    # random nets for convex-point completeness vs brute-force (original: 200)

# --- paired-bootstrap power / MDE (Stage 3) ---
POWER_B              = 600    # bootstrap resamples         (original: 600)
POWER_SIMS           = 400    # Monte-Carlo simulations     (original: 400)
POWER_CAP_N          = 600    # cap on simulated N          (original: 600)
POWER_BIG            = 200000 # normal-approx copula sample (original: 200000)

# --- pre-registered thresholds + funnel constants (verbatim from method.py) ---
PRE_REG = {
    "applicability_number": ">=10% general; 5-10% module; <5% niche",
    "applicability_general_threshold": 0.10,
    "applicability_module_threshold": 0.05,
    "min_effect_size": 0.10,
    "power_target": 0.80,
    "evaluable_definition": "gold singleton edge AND both endpoints appear in >=1 OTHER gold edge (degree>=2)",
    "deduction_required_proxy": "sentence distance > 1 (MATRES via qiangning SENTDIFF; "
                                "TDDMan all-non-local by construction; NarrativeTime |sent(u)-sent(v)|>1)",
    "multipath_definition": ">=2 distinct length-2 intermediates, OR >=1 intermediate AND a "
                            ">=3-edge/cyclic path (bounded DFS, path length cap=4)",
    "primary_recovery_arm": "convex POINT (EXACT, PC complete); ALLEN arm is a sound LOWER BOUND",
    "method": "FULL iterated PC-2 closure", "baseline": "NAIVE single-pass length-2 intersection",
}
PATH_CAP = 4
MAX_EXAMPLES_PER_ARM = 220


## 5. The QCN path-consistency engine (`engine.py`, verbatim)

Bitmask-encoded `Algebra` (loaded from the qualreas tables), a `QCN` constraint network, and the three closure variants used as method vs. baselines: **`pc2_full`** (FULL iterated Mackworth PC-2 — OUR METHOD), **`naive_single_pass`** (single-pass length-2 intersection — BASELINE), and **`closure_off`** (no propagation — lower baseline).

In [ ]:
#!/usr/bin/env python3
"""Reusable QCN (Qualitative Constraint Network) path-consistency engine.

Implements:
  * Algebra        -- one per qualitative calculus (Allen-13, convex Linear Point, RCC-8),
                      loaded from the AUTHORITATIVE qualreas Algebras/*.json composition/converse
                      tables (github.com/alreich/qualreas). Relation sets are encoded as integer
                      bitmasks for speed (compose = OR over base*base table lookups).
  * QCN            -- a constraint network (n x n matrix of relation-set bitmasks).
  * pc2_full       -- Mackworth PC-2 worklist closure to FIXPOINT (OUR METHOD: iterated
                      path-consistency). Returns the closed network + a consistency flag +
                      a fired-composition log.
  * naive_single_pass -- BASELINE: a single pass of length-2 path intersections at the query
                      node only, NO fixpoint, NO propagation ("Path-of-Thoughts + one
                      intersection step").
  * closure_off    -- LOWER BASELINE: table held fixed but closure not run (returns universe).

This module is checkpointed for reuse by later tiers; it has no LLM / corpus dependencies.
"""
from __future__ import annotations

import json
from collections import deque
from pathlib import Path
from typing import Iterable

# --------------------------------------------------------------------------------------
# Algebra
# --------------------------------------------------------------------------------------


class Algebra:
    """A qualitative calculus with bitmask-encoded relation sets.

    base_relations[k] occupies bit (1 << k). A relation SET is the OR of its members' bits.
    universe = all bits set ("no information"). identity = the reflexive/equality relation set.
    """

    def __init__(self, name: str, base_relations: list[str],
                 converse: dict[str, str], compose_tbl: dict[tuple[str, str], frozenset],
                 identity: frozenset):
        self.name = name
        self.base_relations = list(base_relations)
        self.n = len(self.base_relations)
        self.bit = {r: (1 << k) for k, r in enumerate(self.base_relations)}
        self.rel_of_bitpos = {k: r for k, r in enumerate(self.base_relations)}
        self.universe = (1 << self.n) - 1
        self.empty = 0
        self.identity = self.mask_of(identity)
        # converse as a base->bit map for fast set converse
        self._converse_base_bit = {self.bit[r]: self.bit[converse[r]] for r in self.base_relations}
        self.converse_base = dict(converse)
        # base x base composition, as bitmasks indexed by bit position
        self._compose_bb = [[0] * self.n for _ in range(self.n)]
        for (a, b), res in compose_tbl.items():
            self._compose_bb[self.base_relations.index(a)][self.base_relations.index(b)] = self.mask_of(res)
        # caches
        self._compose_cache: dict[tuple[int, int], int] = {}
        self._converse_cache: dict[int, int] = {}

    # ---- set helpers -------------------------------------------------------------
    def mask_of(self, rels: Iterable[str]) -> int:
        m = 0
        for r in rels:
            m |= self.bit[r]
        return m

    def rels_of(self, mask: int) -> list[str]:
        return [self.base_relations[k] for k in range(self.n) if mask & (1 << k)]

    def label(self, mask: int) -> str:
        """Canonical sorted string label, e.g. '<' or 'B|M|O' or 'universe'."""
        if mask == self.universe:
            return "universe"
        if mask == 0:
            return "EMPTY"
        return "|".join(self.rels_of(mask))

    def is_singleton(self, mask: int) -> bool:
        return mask != 0 and (mask & (mask - 1)) == 0

    # ---- algebra operations ------------------------------------------------------
    def converse(self, mask: int) -> int:
        c = self._converse_cache.get(mask)
        if c is not None:
            return c
        out = 0
        m = mask
        while m:
            low = m & (-m)
            out |= self._converse_base_bit[low]
            m ^= low
        self._converse_cache[mask] = out
        return out

    def compose(self, a: int, b: int) -> int:
        """Composition of two relation sets (over-approximation in incomplete algebras)."""
        if a == 0 or b == 0:
            return 0
        if a == self.universe or b == self.universe:
            return self.universe
        key = (a, b)
        c = self._compose_cache.get(key)
        if c is not None:
            return c
        out = 0
        cbb = self._compose_bb
        ma = a
        while ma:
            lowa = ma & (-ma)
            ia = lowa.bit_length() - 1
            row = cbb[ia]
            mb = b
            while mb:
                lowb = mb & (-mb)
                ib = lowb.bit_length() - 1
                out |= row[ib]
                mb ^= lowb
                if out == self.universe:
                    break
            ma ^= lowa
            if out == self.universe:
                break
        self._compose_cache[key] = out
        return out


def _parse_qualreas_relset(s: str) -> frozenset:
    return frozenset(part for part in s.split("|") if part)


def load_algebra_from_qualreas(json_path: str | Path, name: str | None = None) -> Algebra:
    """Parse a qualreas Algebras/*.json file into an Algebra (authoritative tables)."""
    data = json.loads(Path(json_path).read_text())
    rels = data["Relations"]
    base_relations = list(rels.keys())
    converse = {r: rels[r]["Converse"] for r in base_relations}
    identity = frozenset(r for r in base_relations if rels[r].get("Reflexive", False))
    tt = data["TransTable"]
    compose_tbl: dict[tuple[str, str], frozenset] = {}
    for a in base_relations:
        for b in base_relations:
            compose_tbl[(a, b)] = _parse_qualreas_relset(tt[a][b])
    return Algebra(name or data.get("Name", "algebra"), base_relations, converse, compose_tbl, identity)


# --------------------------------------------------------------------------------------
# QCN
# --------------------------------------------------------------------------------------


class QCN:
    """Qualitative constraint network over a fixed node list, edges as relation-set bitmasks.

    Stored as a dense n x n list-of-lists of integer masks. Missing edge => universe.
    Invariants maintained on set_edge: M[j][i] == converse(M[i][j]); M[i][i] == identity.
    """

    def __init__(self, alg: Algebra, nodes: list):
        self.alg = alg
        self.nodes = list(nodes)
        self.n = len(self.nodes)
        self.index = {nd: i for i, nd in enumerate(self.nodes)}
        U = alg.universe
        self.M = [[U] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.M[i][i] = alg.identity
        # neighbours = nodes with a non-universe (informative) edge
        self.nbrs: list[set] = [set() for _ in range(self.n)]

    def set_edge(self, i: int, j: int, mask: int) -> None:
        if i == j:
            return
        self.M[i][j] = mask
        self.M[j][i] = self.alg.converse(mask)
        if mask != self.alg.universe:
            self.nbrs[i].add(j)
            self.nbrs[j].add(i)
        else:
            self.nbrs[i].discard(j)
            self.nbrs[j].discard(i)

    def get(self, i: int, j: int) -> int:
        return self.M[i][j]

    def copy(self) -> "QCN":
        q = QCN.__new__(QCN)
        q.alg = self.alg
        q.nodes = self.nodes
        q.n = self.n
        q.index = self.index
        q.M = [row[:] for row in self.M]
        q.nbrs = [set(s) for s in self.nbrs]
        return q

    def known_edges(self) -> list[tuple[int, int]]:
        U = self.alg.universe
        out = []
        for i in range(self.n):
            for j in range(i + 1, self.n):
                if self.M[i][j] != U:
                    out.append((i, j))
        return out


# --------------------------------------------------------------------------------------
# Closure variants
# --------------------------------------------------------------------------------------


def pc2_full(qcn: QCN, record_fired: bool = False):
    """OUR METHOD: Mackworth PC-2 worklist closure to fixpoint.

    Returns (qcn, consistent: bool, fired_log). On detecting an empty edge the network is
    inconsistent (MODE-B certificate); we stop and report consistent=False.
    Exploits sparsity: only triangles with two informative sides can tighten an edge.
    """
    alg = qcn.alg
    U = alg.universe
    M = qcn.M
    nbrs = qcn.nbrs
    compose = alg.compose
    converse = alg.converse

    Q = deque()
    inq = set()
    for (i, j) in qcn.known_edges():
        Q.append((i, j)); inq.add((i, j))
        Q.append((j, i)); inq.add((j, i))
    fired = [] if record_fired else None

    def enqueue(a, b):
        if (a, b) not in inq:
            inq.add((a, b)); Q.append((a, b))

    while Q:
        i, j = Q.popleft(); inq.discard((i, j))
        rij = M[i][j]
        if rij == U:
            continue
        # path i -> j -> k  tightens (i,k)
        for k in list(nbrs[j]):
            if k == i:
                continue
            comp = compose(rij, M[j][k])
            old = M[i][k]
            new = old & comp
            if new != old:
                if new == 0:
                    return qcn, False, fired
                M[i][k] = new
                M[k][i] = converse(new)
                nbrs[i].add(k); nbrs[k].add(i)
                if fired is not None:
                    fired.append((qcn.nodes[i], qcn.nodes[k], qcn.nodes[j]))
                enqueue(i, k); enqueue(k, i)
        # path k -> i -> j  tightens (k,j)
        for k in list(nbrs[i]):
            if k == j:
                continue
            comp = compose(M[k][i], rij)
            old = M[k][j]
            new = old & comp
            if new != old:
                if new == 0:
                    return qcn, False, fired
                M[k][j] = new
                M[j][k] = converse(new)
                nbrs[k].add(j); nbrs[j].add(k)
                if fired is not None:
                    fired.append((qcn.nodes[k], qcn.nodes[j], qcn.nodes[i]))
                enqueue(k, j); enqueue(j, k)
    return qcn, True, fired


def naive_single_pass(qcn: QCN, u: int, v: int) -> int:
    """BASELINE: single pass of length-2 path compositions arriving at the query edge (u,v).

    Intersects, over every distinct intermediate w with gold edges u-w and w-v, the
    composition compose(R(u,w), R(w,v)). NO fixpoint, NO propagation to other edges,
    NO converse seeding beyond the gold edges already present in `qcn`.
    """
    alg = qcn.alg
    U = alg.universe
    M = qcn.M
    compose = alg.compose
    R = U
    for w in qcn.nbrs[u]:
        if w == v or w == u:
            continue
        if M[w][v] != U:                      # gold length-2 path u - w - v
            R &= compose(M[u][w], M[w][v])
            if R == 0:
                return 0
    return R


def closure_off(qcn: QCN, u: int, v: int) -> int:
    """LOWER BASELINE: table held fixed but closure NOT run -> no information."""
    return qcn.alg.universe


## 6. Engine unit-test battery (`tests.py`, verbatim) — Stage 0 gate

Validates the Allen 169-cell table against published canonical cells + the composition-converse algebra law, confirms convex-point PC **completeness** vs brute-force enumeration, detects an inconsistent cycle, and isolates iteration (FULL==NAIVE on a length-2 query, FULL!=NAIVE on a 3-hop chain). All gating tests must pass before any T0 scoring.

In [ ]:
#!/usr/bin/env python3
"""Engine unit-test battery. ALL must pass before any T0 scoring (Stage 0 gate).

T-A   : loaded Allen 13x13 table matches published Allen-1983 canonical cells + the
        composition/converse algebra law holds for all 169 cells (structural validation
        of every cell against the relation-algebra axioms).
T-P   : convex linear point compositions and converses.
T-C   : consistency detection -- A<B<C<A is inconsistent.
T-CMP : convex point COMPLETENESS -- PC minimal labels == brute-force enumeration of all
        consistent point scenarios on random small nets (n<=6). Confirms PC is complete
        (computes the minimal network) on the convex point algebra.
T-IT  : iteration isolation -- FULL==NAIVE on a length-2 acyclic query, FULL!=NAIVE on a
        3-hop chain (full resolves a singleton naive leaves loose). Underpins the H3
        iterated-vs-single-pass envelope.
T-NEG : (documented) PC is INCOMPLETE on a point net containing the non-convex relation
        '!=' ({<,>}) -- motivates widening non-convex inputs to universe.
"""
from __future__ import annotations

import itertools
import random
from pathlib import Path

# (engine objects Algebra, QCN, load_algebra_from_qualreas, pc2_full, naive_single_pass
#  are defined in the engine cell above)

# ALG_DIR is defined in the data-loading cell (points at the bundled algebra tables).

# Canonical Allen compositions (Allen, CACM 1983 / standard interval-algebra references).
# Keys use qualreas relation symbols: B BI D DI E F FI M MI O OI S SI.
ALLEN_CANONICAL = {
    ("B", "B"): {"B"},
    ("B", "M"): {"B"},
    ("M", "M"): {"B"},
    ("B", "D"): {"B", "O", "M", "D", "S"},
    ("B", "DI"): {"B"},
    ("D", "B"): {"B"},
    ("D", "D"): {"D"},
    ("DI", "DI"): {"DI"},
    ("O", "O"): {"B", "M", "O"},
    ("S", "S"): {"S"},
    ("F", "F"): {"F"},
    ("M", "MI"): {"E", "F", "FI"},
    ("MI", "M"): {"E", "S", "SI"},
    ("S", "SI"): {"E", "S", "SI"},
    ("F", "FI"): {"E", "F", "FI"},
    ("B", "BI"): {"B", "BI", "D", "DI", "E", "F", "FI", "M", "MI", "O", "OI", "S", "SI"},
    ("D", "DI"): {"B", "BI", "D", "DI", "E", "F", "FI", "M", "MI", "O", "OI", "S", "SI"},
}


def _set(alg: Algebra, mask: int) -> set[str]:
    return set(alg.rels_of(mask))


def test_allen(alg: Algebra) -> dict:
    res = {"n_relations": alg.n, "n_cells": alg.n * alg.n}
    assert alg.n == 13, f"Allen must have 13 base relations, got {alg.n}"
    # (a) canonical published cells
    mism = []
    for (a, b), expect in ALLEN_CANONICAL.items():
        got = _set(alg, alg.compose(alg.bit[a], alg.bit[b]))
        if got != expect:
            mism.append((a, b, sorted(got), sorted(expect)))
    res["canonical_cells_checked"] = len(ALLEN_CANONICAL)
    res["canonical_mismatches"] = mism
    # (b) converse involution
    inv_ok = all(alg.converse(alg.converse(alg.bit[r])) == alg.bit[r] for r in alg.base_relations)
    res["converse_involution"] = inv_ok
    # (c) composition-converse algebra law: conv(a o b) == conv(b) o conv(a), all 169 cells
    law_fail = 0
    for a in alg.base_relations:
        for b in alg.base_relations:
            lhs = alg.converse(alg.compose(alg.bit[a], alg.bit[b]))
            rhs = alg.compose(alg.converse(alg.bit[b]), alg.converse(alg.bit[a]))
            if lhs != rhs:
                law_fail += 1
    res["composition_converse_law_failures"] = law_fail
    # (d) identity: E o x == x and x o E == x
    id_ok = all(alg.compose(alg.identity, alg.bit[r]) == alg.bit[r]
                and alg.compose(alg.bit[r], alg.identity) == alg.bit[r]
                for r in alg.base_relations)
    res["identity_law"] = id_ok
    res["passed"] = (not mism) and inv_ok and law_fail == 0 and id_ok
    return res


def test_point(alg: Algebra) -> dict:
    b = alg.bit
    checks = {
        "< o < = <": _set(alg, alg.compose(b["<"], b["<"])) == {"<"},
        "< o = = <": _set(alg, alg.compose(b["<"], b["="])) == {"<"},
        "< o > = univ": alg.compose(b["<"], b[">"]) == alg.universe,
        "> o > = >": _set(alg, alg.compose(b[">"], b[">"])) == {">"},
        "= o = = =": _set(alg, alg.compose(b["="], b["="])) == {"="},
        "conv(<) = >": alg.converse(b["<"]) == b[">"],
        "conv(=) = =": alg.converse(b["="]) == b["="],
        "identity": all(alg.compose(alg.identity, b[r]) == b[r] for r in alg.base_relations),
    }
    return {"checks": checks, "passed": all(checks.values())}


def test_consistency(alg_point: Algebra) -> dict:
    """A<B, B<C, C<A must be inconsistent."""
    q = QCN(alg_point, ["A", "B", "C"])
    lt = alg_point.bit["<"]
    q.set_edge(0, 1, lt)  # A<B
    q.set_edge(1, 2, lt)  # B<C
    q.set_edge(2, 0, lt)  # C<A
    _, consistent, _ = pc2_full(q)
    # positive control: a consistent chain stays consistent
    q2 = QCN(alg_point, ["A", "B", "C"])
    q2.set_edge(0, 1, lt); q2.set_edge(1, 2, lt)
    _, cons2, _ = pc2_full(q2)
    return {"cycle_inconsistent": (not consistent), "chain_consistent": cons2,
            "passed": (not consistent) and cons2}


def _brute_force_point_minimal(constraints: dict[tuple[int, int], set[str]], n: int):
    """Ground-truth minimal network for a CONVEX point net by enumerating coord assignments.

    Returns dict[(i,j)] -> set of relations realized across all assignments consistent with
    the constraints, or None for an (i,j) if the net is inconsistent (no assignment).
    """
    def rel(x, y):
        return "<" if x < y else (">" if x > y else "=")
    realized: dict[tuple[int, int], set[str]] = {(i, j): set() for i in range(n) for j in range(n) if i < j}
    any_consistent = False
    for assign in itertools.product(range(n), repeat=n):
        ok = True
        for (i, j), allowed in constraints.items():
            if rel(assign[i], assign[j]) not in allowed:
                ok = False
                break
        if not ok:
            continue
        any_consistent = True
        for i in range(n):
            for j in range(i + 1, n):
                realized[(i, j)].add(rel(assign[i], assign[j]))
    if not any_consistent:
        return None
    return realized


def test_convex_point_completeness(alg_point: Algebra, n_nets: int = 200, seed: int = 13) -> dict:
    """PC minimal labels == brute-force minimal labels on random small CONVEX point nets."""
    rng = random.Random(seed)
    convex_sets = [["<"], ["="], [">"], ["<", "="], ["=", ">"], ["<", "=", ">"]]
    nets_tested = 0
    nets_consistent = 0
    label_mismatches = 0
    consistency_mismatches = 0
    for _ in range(n_nets):
        n = rng.randint(3, 6)
        # build a random partial convex net by assigning each unordered pair a convex constraint
        constraints: dict[tuple[int, int], set[str]] = {}
        q = QCN(alg_point, list(range(n)))
        for i in range(n):
            for j in range(i + 1, n):
                if rng.random() < 0.55:  # leave some edges as universe
                    rels = rng.choice(convex_sets)
                    constraints[(i, j)] = set(rels)
                    q.set_edge(i, j, alg_point.mask_of(rels))
        bf = _brute_force_point_minimal(constraints, n)
        qc = q.copy()
        _, consistent, _ = pc2_full(qc)
        nets_tested += 1
        if (bf is None) != (not consistent):
            consistency_mismatches += 1
            continue
        if bf is None:
            continue  # both agree inconsistent
        nets_consistent += 1
        for i in range(n):
            for j in range(i + 1, n):
                pc_label = _set(alg_point, qc.get(i, j))
                if pc_label != bf[(i, j)]:
                    label_mismatches += 1
    passed = label_mismatches == 0 and consistency_mismatches == 0
    return {"nets_tested": nets_tested, "nets_consistent": nets_consistent,
            "label_mismatches": label_mismatches, "consistency_mismatches": consistency_mismatches,
            "passed": passed}


def test_iteration_isolation(alg_point: Algebra) -> dict:
    """FULL==NAIVE on a length-2 acyclic query; FULL!=NAIVE on a 3-hop chain."""
    lt = alg_point.bit["<"]
    U = alg_point.universe
    # length-2 acyclic: a<b<c, query (a,c)
    q2 = QCN(alg_point, ["a", "b", "c"])
    q2.set_edge(0, 1, lt); q2.set_edge(1, 2, lt)
    naive2 = naive_single_pass(q2.copy(), 0, 2)
    qf2 = q2.copy(); pc2_full(qf2); full2 = qf2.get(0, 2)
    len2_equal = (naive2 == full2 == lt)
    # 3-hop chain: a<b<c<d, query (a,d) -- no length-2 path a-w-d
    q3 = QCN(alg_point, ["a", "b", "c", "d"])
    q3.set_edge(0, 1, lt); q3.set_edge(1, 2, lt); q3.set_edge(2, 3, lt)
    naive3 = naive_single_pass(q3.copy(), 0, 3)
    qf3 = q3.copy(); pc2_full(qf3); full3 = qf3.get(0, 3)
    hop3_diff = (naive3 == U) and (full3 == lt) and (naive3 != full3)
    return {"len2_naive_eq_full": len2_equal, "hop3_naive_neq_full": hop3_diff,
            "len2_label": alg_point.label(full2), "hop3_full_label": alg_point.label(full3),
            "hop3_naive_label": alg_point.label(naive3),
            "passed": len2_equal and hop3_diff}


def test_pc_incomplete_with_neq(alg_point: Algebra) -> dict:
    """DOCUMENTED: PC is incomplete on a net with the non-convex '!=' relation.

    Classic example: x{<,>}y, y{<,>}z, x{<,>}z is path-consistent but the minimal
    network is tighter than what PC reports (PC leaves all three as {<,>}). We confirm
    PC keeps {<,>} (no tightening) while brute-force shows it stays consistent -- the
    incompleteness shows up on larger '!=' nets; here we just confirm PC does not crash
    and leaves '!=' un-tightened, motivating widening non-convex inputs to universe.
    """
    neq = alg_point.bit["<"] | alg_point.bit[">"]
    q = QCN(alg_point, ["x", "y", "z"])
    q.set_edge(0, 1, neq); q.set_edge(1, 2, neq); q.set_edge(0, 2, neq)
    qc = q.copy()
    _, consistent, _ = pc2_full(qc)
    stays_neq = qc.get(0, 1) == neq and consistent
    return {"pc_keeps_neq_unchanged": stays_neq, "consistent": consistent,
            "note": "PC sound but not guaranteed minimal on non-convex (!=) inputs",
            "passed": True}  # documentation test, always informational


def run_all() -> dict:
    allen = load_algebra_from_qualreas(ALG_DIR / "Linear_Interval_Algebra.json", "ALLEN13")
    point = load_algebra_from_qualreas(ALG_DIR / "Linear_Point_Algebra.json", "POINT")
    rcc8 = load_algebra_from_qualreas(ALG_DIR / "RCC8_Algebra.json", "RCC8")
    out = {
        "allen": test_allen(allen),
        "point": test_point(point),
        "consistency": test_consistency(point),
        "convex_point_completeness": test_convex_point_completeness(point, n_nets=N_NETS_COMPLETENESS),
        "iteration_isolation": test_iteration_isolation(point),
        "pc_incompleteness_with_neq": test_pc_incomplete_with_neq(point),
        "rcc8_cells": rcc8.n * rcc8.n,
        "rcc8_n_relations": rcc8.n,
    }
    out["all_gating_tests_passed"] = bool(
        out["allen"]["passed"] and out["point"]["passed"] and out["consistency"]["passed"]
        and out["convex_point_completeness"]["passed"] and out["iteration_isolation"]["passed"]
    )
    return out


## 7. T0 envelope funnel (`method.py`, verbatim)

Per held-out gold edge, the funnel runs `evaluable -> (i) deduction-required -> (ii) multi-path -> (iii) bite-after-widening -> (iv) N*`, computing the FULL (our method) vs NAIVE (baseline) predictions and the iteration envelope (`full_only` = edges FULL resolves that single-pass cannot). A verified complete-graph fast path keeps runtime low on dense gold.

In [ ]:
def build_base_qcn(alg: Algebra, doc: dict):
    """Build the full gold QCN for a document and the undirected gold adjacency + degree."""
    nodes = sorted({e["u"] for e in doc["edges"]} | {e["v"] for e in doc["edges"]})
    q = QCN(alg, nodes)
    adj = defaultdict(set)
    edge_meta = {}
    for e in doc["edges"]:
        ui, vi = q.index[e["u"]], q.index[e["v"]]
        mask = alg.mask_of(e["rels"])
        # if multiple gold edges on same pair, intersect (consistent annotation)
        cur = q.get(ui, vi)
        q.set_edge(ui, vi, cur & mask if cur != alg.universe else mask)
        adj[e["u"]].add(e["v"]); adj[e["v"]].add(e["u"])
        edge_meta[(e["u"], e["v"])] = e
    return q, adj, edge_meta, nodes


def has_long_path(adj: dict, u, v, cap: int = PATH_CAP, max_expand: int = 4000) -> bool:
    """Bounded DFS for a simple path u->v with >=3 edges and <=cap edges (query edge absent from adj)."""
    stack = [(u, (u,))]
    expand = 0
    while stack:
        node, path = stack.pop()
        expand += 1
        if expand > max_expand:
            return False
        if len(path) > cap:
            continue
        for w in adj.get(node, ()):  # neighbours
            if w == v:
                if len(path) >= 3:
                    return True
                continue
            if w in path:
                continue
            if len(path) < cap:
                stack.append((w, path + (w,)))
    return False


def is_nonconvex_point(alg: Algebra, mask: int) -> bool:
    """For the POINT algebra, the only non-convex relation set is {<,>} (the '!=' relation)."""
    if alg.name != "POINT":
        return False
    return mask == (alg.bit["<"] | alg.bit[">"])


def widen_nonconvex(alg: Algebra, q: QCN) -> tuple[QCN, int]:
    """Return a copy with non-convex input edges widened to universe; count how many widened."""
    qw = q.copy()
    changed = 0
    if alg.name == "POINT":
        neq = alg.bit["<"] | alg.bit[">"]
        for i in range(qw.n):
            for j in range(i + 1, qw.n):
                if qw.M[i][j] == neq:
                    qw.set_edge(i, j, alg.universe); changed += 1
    # (Allen non-convex widening is a no-op on singleton-atomic gold inputs by construction)
    return qw, changed


# ======================================================================================
# Stage 2: T0 funnel for one corpus arm
# ======================================================================================

def run_funnel(arm_name: str, alg: Algebra, docs: dict, collect_examples: bool = True,
               node_cap: int = 200):
    t0 = time.time()
    n_eval = 0
    n_gold_singleton = 0
    n_inconsistent = 0
    n_docs_used = 0
    funnel = Counter()
    iter_env = Counter()          # over the i_ii (deduction-required multipath) set
    widening_destroyed = 0
    capped_docs = 0
    examples = []                 # candidate example records (filtered/stratified later)
    U = alg.universe
    # VERIFY_CAP_PER_DOC is taken from the config cell (genuine-PC cross-checks of the fast path)
    n_complete_docs = 0; n_complete_preclosed_ok = 0
    verify_total = 0; verify_mismatch = 0

    for docid, doc in docs.items():
        if not doc["edges"]:
            continue
        q, adj, emeta, nodes = build_base_qcn(alg, doc)
        if q.n > node_cap:
            capped_docs += 1
            continue
        n_docs_used += 1
        deg = {nd: len(adj[nd]) for nd in nodes}
        # completeness: a complete gold graph (every pair a singleton) is already its own
        # transitive closure, so the held-out full-PC value coincides with the single-pass
        # intersection over all intermediates (proven; verified on a per-doc sample below).
        n_pairs = q.n * (q.n - 1) // 2
        complete = (n_pairs > 0 and len(q.known_edges()) == n_pairs)
        complete_ok = False
        if complete:
            qc = q.copy()
            _, cons_pc, _ = pc2_full(qc)
            unchanged = all(qc.M[i][j] == q.M[i][j] for i in range(q.n) for j in range(q.n))
            complete_ok = bool(cons_pc and unchanged)
            n_complete_docs += 1; n_complete_preclosed_ok += int(complete_ok)
            del qc
        doc_nonconvex = sum(1 for i in range(q.n) for j in range(i + 1, q.n)
                            if is_nonconvex_point(alg, q.M[i][j]))
        verify_used = 0
        for (eu, ev), e in emeta.items():
            gmask = alg.mask_of(e["rels"])
            if not alg.is_singleton(gmask):
                continue
            n_gold_singleton += 1
            # evaluable: both endpoints degree >= 2 (each in >=1 OTHER gold edge)
            if deg[eu] < 2 or deg[ev] < 2:
                continue
            n_eval += 1
            ui, vi = q.index[eu], q.index[ev]
            # (i) deduction-required
            ded = e.get("deduction_required")
            if ded is None:
                sd = e.get("sentdiff")
                ded = (sd is not None and sd > 1)
            if not ded:
                continue
            funnel["i"] += 1
            # (ii) multipath -- adjacency with the query edge removed
            adj_u = adj[eu] - {ev}; adj_v = adj[ev] - {eu}
            inter2 = (adj_u & adj_v) - {eu, ev}
            if len(inter2) >= 2:
                ii_pass = True; longer = None
            else:
                adj_q = {k: (set(s) - ({ev} if k == eu else set()) - ({eu} if k == ev else set()))
                         for k, s in adj.items()}
                longer = has_long_path(adj_q, eu, ev)
                ii_pass = (len(inter2) >= 1 and longer)
            if not ii_pass:
                continue
            funnel["i_ii"] += 1
            # build held-out network (gold minus query edge)
            gm = q.copy(); gm.set_edge(ui, vi, U)
            naive_d = naive_single_pass(gm, ui, vi)         # BASELINE (pre-closure, gold only)
            if complete_ok:
                # FAST PATH: on a complete already-closed graph, full PC == single-pass over
                # all intermediates (== naive over all gold neighbours). Verified on a sample.
                full_d = naive_d
                consistent = True
                if verify_used < VERIFY_CAP_PER_DOC:
                    verify_used += 1
                    gmc = gm.copy()
                    gmc_closed, cons2, _ = pc2_full(gmc)
                    fd_gen = gmc_closed.get(ui, vi) if cons2 else U
                    verify_total += 1
                    verify_mismatch += int(fd_gen != full_d)
                    del gmc, gmc_closed
            else:
                gm_closed, consistent, _ = pc2_full(gm)     # OUR METHOD (genuine iterated PC)
                if not consistent:
                    n_inconsistent += 1
                    del gm; continue
                full_d = gm_closed.get(ui, vi)
            resolves_full = (full_d == gmask)
            resolves_naive = (naive_d == gmask)
            full_only = resolves_full and not resolves_naive
            iter_env["full_resolves"] += int(resolves_full)
            iter_env["naive_resolves"] += int(resolves_naive)
            iter_env["full_only"] += int(full_only)
            iter_env["off_resolves"] += int(U == gmask)     # always 0 for non-universal gold
            # (iii) bite after non-convex widening (no-op on atomic singleton inputs)
            if doc_nonconvex == 0:
                derived_w = full_d
            else:
                qw, _ = widen_nonconvex(alg, q)
                qw.set_edge(ui, vi, U)
                qw_closed, cons_w, _ = pc2_full(qw)
                derived_w = qw_closed.get(ui, vi) if cons_w else U
                del qw_closed, qw
            bite = (derived_w != U)
            if not bite:
                widening_destroyed += int(resolves_full)
                del gm; continue
            funnel["i_ii_iii"] += 1
            # (iv) N*
            nstar = (derived_w == gmask)
            funnel["N_star"] += int(nstar)

            if collect_examples:
                examples.append({
                    "docid": docid, "u": eu, "v": ev, "algebra": alg.name,
                    "gold_label": e.get("gold"), "gold_rels": e["rels"],
                    "predict_full": alg.label(full_d), "predict_naive": alg.label(naive_d),
                    "predict_off": "universe",
                    "resolves_full": resolves_full, "resolves_naive": resolves_naive,
                    "full_only": full_only, "n_star": nstar, "bite": bite,
                    "deduction_required": True, "multipath": True,
                    "sentdiff": e.get("sentdiff"), "n_inter2": len(inter2),
                    "longer_path": bool(longer) if longer is not None else None,
                    "edges_neighbourhood": _neighbourhood_edges(alg, q, adj, emeta, eu, ev),
                })
            del gm
        del q, adj, emeta
        gc.collect()

    n_eval_eff = max(n_eval, 1)
    applic_frac = funnel["i_ii_iii"] / n_eval_eff
    applic_frac_alt = funnel["i_ii_iii"] / max(n_gold_singleton, 1)
    band = ("general" if applic_frac >= PRE_REG["applicability_general_threshold"]
            else ("module" if applic_frac >= PRE_REG["applicability_module_threshold"] else "niche"))
    n_ii = max(funnel["i_ii"], 1)
    out = {
        "arm": arm_name, "algebra": alg.name,
        "n_docs": len(docs), "n_docs_used": n_docs_used, "n_docs_capped": capped_docs,
        "n_gold_singleton_edges": n_gold_singleton, "n_evaluable": n_eval,
        "n_inconsistent_holdouts": n_inconsistent,
        "complete_graph_fast_path": {
            "n_complete_docs": n_complete_docs,
            "n_complete_preclosed_consistent_unchanged": n_complete_preclosed_ok,
            "genuine_pc_crosschecks": verify_total,
            "fast_path_vs_genuine_pc_mismatches": verify_mismatch,
            "note": "complete gold graphs use the proven full-PC==single-pass identity; "
                    "mismatches==0 confirms the fast path matches genuine iterated PC.",
        },
        "funnel": {"i_deduction_required": funnel["i"], "i_ii_multipath": funnel["i_ii"],
                   "i_ii_iii_bite": funnel["i_ii_iii"], "N_star": funnel["N_star"]},
        "applicability_fraction": round(applic_frac, 5),
        "applicability_fraction_alt_denominator": round(applic_frac_alt, 5),
        "applicability_band": band,
        "N_star_fraction": round(funnel["N_star"] / n_eval_eff, 5),
        "iteration_envelope": {
            "denominator_i_ii": funnel["i_ii"],
            "full_resolves": iter_env["full_resolves"],
            "naive_resolves": iter_env["naive_resolves"],
            "full_only_ge3edge_or_cyclic": iter_env["full_only"],
            "full_only_fraction_of_eval": round(iter_env["full_only"] / n_eval_eff, 5),
            "full_only_fraction_of_i_ii": round(iter_env["full_only"] / n_ii, 5),
            "full_resolves_fraction_of_i_ii": round(iter_env["full_resolves"] / n_ii, 5),
            "naive_resolves_fraction_of_i_ii": round(iter_env["naive_resolves"] / n_ii, 5),
        },
        "bite_loss": {"widening_destroyed": widening_destroyed},
        "runtime_sec": round(time.time() - t0, 2),
    }
    # invariants (assert + record)
    inv = {
        "funnel_monotone": funnel["i"] >= funnel["i_ii"] >= funnel["i_ii_iii"] >= funnel["N_star"],
        "full_only_le_full_resolves": iter_env["full_only"] <= iter_env["full_resolves"],
        "naive_le_full": iter_env["naive_resolves"] <= iter_env["full_resolves"] + iter_env.get("naive_only", 0),
    }
    out["invariants"] = inv
    if not inv["funnel_monotone"] or not inv["full_only_le_full_resolves"]:
        logger.error(f"INVARIANT VIOLATION in {arm_name}: {inv}")
    logger.info(f"{arm_name}: n_eval={n_eval} i={funnel['i']} i_ii={funnel['i_ii']} "
                f"i_ii_iii={funnel['i_ii_iii']} N*={funnel['N_star']} "
                f"full_only={iter_env['full_only']} applic={applic_frac:.4f} ({band}) "
                f"[{out['runtime_sec']}s]")
    return out, examples


def _neighbourhood_edges(alg, q, adj, emeta, u, v, cap=24):
    """Compact list of gold constraints near the query (edges incident to u or v), for the example input."""
    out = []
    seen = set()
    for (a, b), e in emeta.items():
        if (a in (u, v) or b in (u, v)) and (a, b) != (u, v):
            key = (a, b)
            if key in seen:
                continue
            seen.add(key)
            out.append(f"{a} --{'|'.join(e['rels'])}--> {b}")
            if len(out) >= cap:
                break
    return out


## 8. A-priori power / MDE (`method.py`, verbatim)

Paired-bootstrap power and minimum-detectable-effect at 80% power for the held-out comparison, sizing the iter-2 LLM experiment before any budget is spent.

In [ ]:
def power_mde(N: int, base_acc: float = 0.5, rho: float = 0.3,
              effects=(0.05, 0.10, 0.15, 0.20, 0.25), B: int = 600, sims: int = 400,
              seed: int = 12345, cap_N: int = 600, big: int = 200_000):
    """A-priori paired-bootstrap power: P(95% bootstrap CI on mean paired diff excludes 0).

    Paired binary outcomes via a Gaussian copula (correlation rho) with marginals
    base_acc (baseline) and base_acc+effect (method). cap_N bounds the simulated N for speed
    (power saturates well below it); actual N is reported."""
    from scipy.stats import norm
    res = {"N": N, "base_acc": base_acc, "rho_pair": rho, "B": B, "sims": sims,
           "N_used_in_sim": min(N, cap_N)}
    if N <= 1:
        zero = {f"{e:.2f}": 0.0 for e in effects}
        res.update({"power_by_effect": zero, "power_by_effect_bootstrap_capped_N": zero,
                    "power_by_effect_normal_approx_true_N": zero, "MDE_80": None,
                    "MDE_80_bootstrap_capped_N": None, "MDE_80_normal_approx_true_N": None,
                    "power_at_0.10": 0.0, "power_at_0.20": 0.0})
        return res
    Nb = min(N, cap_N)
    rng = np.random.default_rng(seed)
    z_base = norm.ppf(1 - base_acc)
    power = {}
    for eff in effects:
        acc_m = min(0.999, base_acc + eff)
        z_m = norm.ppf(1 - acc_m)
        Xb = rng.standard_normal((sims, Nb))
        Em = rng.standard_normal((sims, Nb))
        Ym = rho * Xb + np.sqrt(1 - rho**2) * Em
        diffs = (Ym > z_m).astype(np.int8) - (Xb > z_base).astype(np.int8)  # sims x Nb
        wins = 0
        # vectorised bootstrap in chunks of sims to bound memory
        chunk = max(1, min(sims, 4_000_000 // (B * Nb + 1)))
        for s0 in range(0, sims, chunk):
            blk = diffs[s0:s0 + chunk]                       # (c, Nb)
            c = blk.shape[0]
            idx = rng.integers(0, Nb, size=(c, B, Nb))
            boot_means = blk[np.arange(c)[:, None, None], idx].mean(axis=2)  # (c, B)
            lo = np.percentile(boot_means, 2.5, axis=1)      # (c,)
            wins += int((lo > 0).sum())
        power[eff] = wins / sims
    res["power_by_effect_bootstrap_capped_N"] = {f"{e:.2f}": round(power[e], 4) for e in effects}
    res["power_by_effect"] = res["power_by_effect_bootstrap_capped_N"]
    mde = next((e for e in sorted(effects) if power[e] >= 0.80), None)
    res["MDE_80"] = mde
    res["MDE_80_bootstrap_capped_N"] = mde
    res["power_at_0.10"] = round(power.get(0.10, 0.0), 4)
    res["power_at_0.20"] = round(power.get(0.20, 0.0), 4)

    # FAITHFUL large-N power via the normal approximation at the TRUE (uncapped) N.
    # Estimate the per-pair diff sd once per effect from a large copula sample, then
    # power = P(95% CI lower bound > 0) ~ Phi(delta*sqrt(N)/sd - 1.96).
    Xb = rng.standard_normal(big)
    Em = rng.standard_normal(big)
    Ym = rho * Xb + np.sqrt(1 - rho**2) * Em
    base_succ = (Xb > z_base).astype(np.float64)
    pna = {}
    for eff in effects:
        z_m = norm.ppf(1 - min(0.999, base_acc + eff))
        d = (Ym > z_m).astype(np.float64) - base_succ
        delta = d.mean(); sd = d.std(ddof=1)
        if sd <= 0:
            pna[eff] = 1.0 if delta > 0 else 0.0
        else:
            pna[eff] = float(norm.cdf(delta * np.sqrt(N) / sd - 1.96))
    res["power_by_effect_normal_approx_true_N"] = {f"{e:.2f}": round(pna[e], 4) for e in effects}
    res["MDE_80_normal_approx_true_N"] = next((e for e in sorted(effects) if pna[e] >= 0.80), None)
    return res


## 9. Example assembly helpers (`method.py`, verbatim)

`stratified_examples` turns funnel candidates into per-query rows (input + gold + method/baseline predictions), prioritising the scientifically interesting `full_only` / `N*` rows.

In [ ]:
def stratified_examples(arm_name: str, corpus: str, cand: list, cap: int = MAX_EXAMPLES_PER_ARM):
    """Prioritise scientifically interesting rows (N*+, full_only) then a deterministic sample."""
    full_only = [c for c in cand if c["full_only"]]
    nstar = [c for c in cand if c["n_star"] and not c["full_only"]]
    rest = [c for c in cand if not c["n_star"] and not c["full_only"]]
    rest.sort(key=lambda c: (c["docid"], str(c["u"]), str(c["v"])))
    chosen = full_only[:cap]
    if len(chosen) < cap:
        chosen += nstar[:cap - len(chosen)]
    if len(chosen) < cap and rest:
        step = max(1, len(rest) // max(1, (cap - len(chosen))))
        chosen += rest[::step][:cap - len(chosen)]
    rows = []
    for c in chosen:
        body = "\n".join("  " + s for s in c["edges_neighbourhood"])
        inp = (f"Temporal qualitative-reasoning query [corpus={corpus}, doc={c['docid']}, "
               f"algebra={c['algebra']}].\n"
               f"Held-out query: relation({c['u']}, {c['v']}) = ?\n"
               f"Known gold constraints (query edge removed):\n{body}\n"
               f"Task: derive relation({c['u']},{c['v']}) by qualitative constraint propagation "
               f"(path-consistency). Answer with the tightest relation set.")
        rows.append({
            "input": inp,
            "output": str(c["gold_label"]),
            "predict_our_method_full_pc": c["predict_full"],
            "predict_baseline_naive_singlepass": c["predict_naive"],
            "predict_baseline_closure_off": c["predict_off"],
            "metadata_corpus": corpus,
            "metadata_docid": c["docid"],
            "metadata_query_u": str(c["u"]),
            "metadata_query_v": str(c["v"]),
            "metadata_gold_rels": c["gold_rels"],
            "metadata_deduction_required": c["deduction_required"],
            "metadata_multipath": c["multipath"],
            "metadata_n_intermediates_len2": c["n_inter2"],
            "metadata_has_long_path": c["longer_path"],
            "metadata_sentence_distance": c["sentdiff"],
            "metadata_resolves_full_pc": c["resolves_full"],
            "metadata_resolves_naive": c["resolves_naive"],
            "metadata_iteration_dependent_full_only": c["full_only"],
            "metadata_in_N_star": c["n_star"],
        })
    return rows, {"n_candidates": len(cand), "n_emitted": len(rows),
                  "n_full_only_in_pool": len(full_only), "n_nstar_in_pool": len([c for c in cand if c["n_star"]])}


def _json_default(o):
    if isinstance(o, (set, frozenset)):
        return sorted(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    return str(o)


## 10. Run the pipeline

Stage 0 tests -> Stage 1 load gold graphs -> Stage 2 funnel per arm -> Stage 3 power -> Stage 4 non-circularity split. (Mirrors `method.main()`; the only change is that Stage 1 loads pre-parsed gold graphs instead of re-parsing the raw corpus.)

In [ ]:
def run_demo():
    """Stage 0 tests -> Stage 1 load gold graphs -> Stage 2 funnel -> Stage 3 power -> Stage 4 split.
    (Mirrors method.main(); the raw-corpus parse of Stage 1 is replaced by loading pre-parsed graphs.)"""
    overall_t0 = time.time()

    logger.info("=== STAGE 0: engine unit-test battery (gates everything) ===")
    eng_val = run_all()
    assert eng_val["all_gating_tests_passed"], "ENGINE GATING TESTS FAILED -- aborting before T0 scoring"
    logger.info("engine gating tests PASSED")

    logger.info("=== STAGE 1: load pre-parsed gold graphs (parsers.py output) ===")
    allen = load_algebra_from_qualreas(ALG_DIR / "Linear_Interval_Algebra.json", "ALLEN13")
    point = load_algebra_from_qualreas(ALG_DIR / "Linear_Point_Algebra.json", "POINT")

    arms_spec = [
        ("NarrativeTime_point", point, "NarrativeTime"),   # convex POINT arm -> EXACT (PC complete)
        ("NarrativeTime_allen", allen, "NarrativeTime"),   # ALLEN interval arm -> sound LOWER BOUND
    ]

    logger.info("=== STAGE 2: T0 envelope funnel per arm ===")
    per_arm = {}; examples_by_arm = {}
    for arm_name, alg, corpus in arms_spec:
        arm = load_arm(arm_name)
        docs = dict(list(arm.get("docs", {}).items())[:MAX_DOCS_PER_ARM])
        out, cand = run_funnel(arm_name, alg, docs, node_cap=NODE_CAP)
        out["corpus"] = corpus
        per_arm[arm_name] = out
        examples_by_arm[arm_name] = (corpus, cand)

    logger.info("=== STAGE 3: paired-bootstrap power / MDE ===")
    for arm_name, out in per_arm.items():
        N = out["funnel"]["i_ii_iii_bite"]
        out["power"] = power_mde(N, B=POWER_B, sims=POWER_SIMS, cap_N=POWER_CAP_N, big=POWER_BIG)
        logger.info(f"{arm_name}: power N={N} MDE_80={out['power']['MDE_80']} "
                    f"p@.10={out['power']['power_at_0.10']} p@.20={out['power']['power_at_0.20']}")

    logger.info("=== STAGE 4: NarrativeTime non-circularity split ===")
    nt_arm = load_arm("NarrativeTime_point")
    nt_docs = dict(list(nt_arm.get("docs", {}).items())[:MAX_DOCS_PER_ARM])
    loc_just = 0; timeline_implied = 0; total_nt = 0
    for doc in nt_docs.values():
        for e in doc["edges"]:
            sd = e.get("sentdiff")
            if sd is None:
                continue
            total_nt += 1
            if sd <= 1:
                loc_just += 1
            else:
                timeline_implied += 1
    nt_noncirc = {
        "locally_justifiable_frac": round(loc_just / max(total_nt, 1), 5),
        "purely_timeline_implied_frac": round(timeline_implied / max(total_nt, 1), 5),
        "n_edges": total_nt,
        "headline_subset": "purely_timeline_implied (sentence distance > 1)",
    }
    per_arm["NarrativeTime_point"]["narrativetime_noncircularity"] = nt_noncirc

    logger.info(f"=== demo complete in {round(time.time() - overall_t0, 2)}s ===")
    return per_arm, examples_by_arm, eng_val


per_arm, examples_by_arm, eng_val = run_demo()


## 11. Results & visualization

Engine-validation summary, the per-arm T0 funnel table (applicability band, N* recovery, FULL-vs-NAIVE iteration envelope, power/MDE), the funnel + power-curve plots, and a few per-query method-vs-baseline predictions.

In [ ]:
# ============================ RESULTS SUMMARY + VISUALIZATION ============================

# --- engine validation (Stage 0 gate) ---
print("ENGINE VALIDATION (Stage 0 gate):")
print(f"  Allen 169-cell table valid (canonical cells + composition-converse law): "
      f"{eng_val['allen']['passed']} "
      f"[law failures={eng_val['allen']['composition_converse_law_failures']}, "
      f"canonical cells checked={eng_val['allen']['canonical_cells_checked']}]")
cpc = eng_val['convex_point_completeness']
print(f"  Convex point PC complete vs brute-force: {cpc['passed']} "
      f"[nets tested={cpc['nets_tested']}, label mismatches={cpc['label_mismatches']}]")
it = eng_val['iteration_isolation']
print(f"  Iteration isolation (FULL==NAIVE on len-2; FULL!=NAIVE on 3-hop): {it['passed']} "
      f"[len-2 -> {it['len2_label']}, 3-hop FULL -> {it['hop3_full_label']}, NAIVE -> {it['hop3_naive_label']}]")
print(f"  ALL gating tests passed: {eng_val['all_gating_tests_passed']}")
print()

# --- per-arm funnel / applicability / N* / power table ---
def summarize(arm_name, exact):
    a = per_arm[arm_name]; ie = a["iteration_envelope"]; f = a["funnel"]
    return {
        "arm": arm_name, "recovery": "EXACT" if exact else "LOWER_BOUND",
        "n_eval": a["n_evaluable"], "applic": a["applicability_fraction"], "band": a["applicability_band"],
        "i": f["i_deduction_required"], "i_ii": f["i_ii_multipath"],
        "i_ii_iii": f["i_ii_iii_bite"], "N*": f["N_star"],
        "full_res": ie["full_resolves"], "naive_res": ie["naive_resolves"],
        "full_only": ie["full_only_ge3edge_or_cyclic"],
        "MDE80": a["power"]["MDE_80"], "p@.10": a["power"]["power_at_0.10"],
    }

rows = [summarize("NarrativeTime_point", True), summarize("NarrativeTime_allen", False)]
cols = list(rows[0].keys())
w = {c: max(len(str(c)), *(len(str(r[c])) for r in rows)) for c in cols}
print("PER-ARM T0 ENVELOPE FUNNEL (held-out gold edges):")
print("  " + "  ".join(str(c).ljust(w[c]) for c in cols))
for r in rows:
    print("  " + "  ".join(str(r[c]).ljust(w[c]) for c in cols))
print()
nc = per_arm["NarrativeTime_point"]["narrativetime_noncircularity"]
print(f"NarrativeTime non-circularity split: {nc['purely_timeline_implied_frac']:.3f} of edges are "
      f"purely-timeline-implied (sentdiff>1), {nc['locally_justifiable_frac']:.3f} locally justifiable "
      f"(n={nc['n_edges']}).")

# --- plots: the T0 funnel (point arm) + paired-bootstrap power curve ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
a = per_arm["NarrativeTime_point"]; f = a["funnel"]
stages = ["evaluable", "(i) deduction\nrequired", "(ii) multi\npath", "(iii) bite", "(iv) N* exact"]
vals = [a["n_evaluable"], f["i_deduction_required"], f["i_ii_multipath"], f["i_ii_iii_bite"], f["N_star"]]
axes[0].bar(range(len(stages)), vals, color="#3b7dd8")
axes[0].set_xticks(range(len(stages))); axes[0].set_xticklabels(stages, fontsize=8)
axes[0].set_title("NarrativeTime point arm: T0 funnel"); axes[0].set_ylabel("held-out gold edges")
for i, v in enumerate(vals):
    axes[0].text(i, v, str(v), ha="center", va="bottom", fontsize=8)

pw = a["power"]["power_by_effect"]
effs = sorted(float(k) for k in pw); ps = [pw[f"{e:.2f}"] for e in effs]
axes[1].plot(effs, ps, "o-", color="#d8533b")
axes[1].axhline(0.80, ls="--", c="gray", lw=1, label="power = 0.80")
axes[1].axvline(0.10, ls=":", c="green", lw=1, label="min effect 0.10")
axes[1].set_ylim(-0.02, 1.05)
axes[1].set_xlabel("effect size (accuracy gain)"); axes[1].set_ylabel("power")
axes[1].set_title(f"Paired-bootstrap power (N={f['i_ii_iii_bite']})"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

# --- a few per-query method-vs-baseline predictions ---
corpus, cand = examples_by_arm["NarrativeTime_point"]
rows_ex, _ = stratified_examples("NarrativeTime_point", corpus, cand)
print(f"\nExample held-out queries (showing {min(3, len(rows_ex))} of {len(rows_ex)} emitted):")
for r in rows_ex[:3]:
    print("-" * 78)
    print(r["input"][:260])
    print(f"  GOLD={r['output']}  OUR(full_pc)={r['predict_our_method_full_pc']}  "
          f"NAIVE={r['predict_baseline_naive_singlepass']}  OFF={r['predict_baseline_closure_off']}")
